# Jinja
Jinja ist eine Python Renderer Engine und könnte dazu genutzt werden die Implementierung der Transformation in ein Template einzufügen, um so eine leichter zu überprüfende Datei zu erhalten.

In [1]:
from jinja2 import Environment, FileSystemLoader

env = Environment(loader=FileSystemLoader('templates'))
template = env.get_template("test.jinja")
output = template.render()
print(output)

public class Test {
    public static void main(String[] args) {
        var surprise = new Test().surprise();
        System.out.println("Hello, World!", surprise);
    }

    public String surprise() {
        
    }
}


Anschließend soll die Methode surprise KI-generierter Code eingefügt werden. Der Output soll dann wiefolgt aussehen:

In [2]:
output = template.render(surprise_code='return "This is a surprise!";')
print(output)

public class Test {
    public static void main(String[] args) {
        var surprise = new Test().surprise();
        System.out.println("Hello, World!", surprise);
    }

    public String surprise() {
        return "This is a surprise!";
    }
}


Als Input in das LLM soll dabei das Template geschickt werden.

In [2]:
from pathlib import Path
import sys
import logging

logging.basicConfig(level=logging.INFO)

ROOT = Path.cwd().parent.parent
logging.info(f"Adding {ROOT} to Python path for imports in kernel.")
sys.path.insert(0, str(ROOT))  # ROOT, nicht SRC!

# Load config with changed env path
from bxagent.config import Config
config = Config.get_instance(env_path=ROOT / ".env")


INFO:root:Adding /Users/lukas/Masterarbeit/agents/bxAgent to Python path for imports in kernel.


In [3]:
from bxagent.models import build_base_model

model = build_base_model()

In [20]:
blank_template = template.render()
print(blank_template)
response = model.invoke(f"Fill out the surprise function in Java Code. Just answer with the code body of the function, no explanations. The template is:\n\n{blank_template}")

public class Test {
    public static void main(String[] args) {
        var surprise = new Test().surprise();
        System.out.println("Hello, World!", surprise);
    }

    public String surprise() {
        
    }
}


INFO:httpx:HTTP Request: POST https://chat-1.ki-awz.iisys.de/api/chat/completions "HTTP/1.1 200 OK"


In [21]:
print(response)

content='\n\nreturn "Surprise!";' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 4957, 'prompt_tokens': 91, 'total_tokens': 5048, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'lisa-pro', 'system_fingerprint': None, 'id': 'chatcmpl-74eff647-0334-489b-b075-d08ae6dd651a', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019ea6da-d3fa-7ce2-b975-30fc58749545-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 91, 'output_tokens': 4957, 'total_tokens': 5048, 'input_token_details': {}, 'output_token_details': {}}


Danach muss der Content in den Renderer eingefügt werden.

In [22]:
output = template.render(surprise_code=response.content)
print(output)

public class Test {
    public static void main(String[] args) {
        var surprise = new Test().surprise();
        System.out.println("Hello, World!", surprise);
    }

    public String surprise() {
        

return "Surprise!";
    }
}


Hier sieht man direkt das erste Problem. Wie wird wirklich sichergestellt, dass der Output **immer** das Ergebnis hat?

Hier könnte man abschließend noch einen Formatierer drüberlaufen lassen, damit das Format wiederhergestellt werden kann.

In [5]:
base_template = template.render(surprise_code='// TODO: Implement surprise function')

def test_renderer_output(run_id: int):
    response = model.invoke(f"Fill out the surprise function in Java Code. Just answer with the code body of the function, no explanations. The template is:\n\n{base_template}")
    output = template.render(surprise_code=response.content)
    with open(f"test_results/output_{run_id}.java", "w") as f:
        f.write(output)

In [8]:
threads = []
for i in range(25):
    test_renderer_output(i)

INFO:httpx:HTTP Request: POST https://chat-1.ki-awz.iisys.de/api/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-1.ki-awz.iisys.de/api/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-1.ki-awz.iisys.de/api/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-1.ki-awz.iisys.de/api/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-1.ki-awz.iisys.de/api/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-1.ki-awz.iisys.de/api/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-1.ki-awz.iisys.de/api/chat/completions "HTTP/1.1 504 Gateway Timeout"
INFO:openai._base_client:Retrying request to /chat/completions in 0.477346 seconds
INFO:httpx:HTTP Request: POST https://chat-1.ki-awz.iisys.de/api/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-1.ki-awz.iisys.de/api/chat/completions "HTTP/1.1 200 OK"


KeyboardInterrupt: 

Das hat teilweise unterschiedliche Ergebnisse geliefert. Einmal nur den Javacode und einmal gewrapt in ```java Blöcken. D.h. hier sollte ich den Prompt anpassen.

In [13]:
prompt = """
Below is a template with empty function bodies.

--- BEGIN TEMPLATE ---
{template}
--- END TEMPLATE ---

Fill out the function body based on the function name and comments. Always wrap your answer in a code block, and only provide the code block, no explanations.

Example:
```java
// This function calculates the factorial of a number
return (n == 0) ? 1 : n * factorial(n-1);
```
"""

In [ ]:
base_template = template.render(surprise_code='// TODO: Implement surprise function')
print(base_template)
input_msg = prompt.format(template=base_template)

def test_approach_2(run_id: int):
    response = model.invoke(input_msg)
    with open(f"test_results/approach_2_output_{run_id}.txt", "w") as f:
        f.write(response.content)

public class Test {
    public static void main(String[] args) {
        var surprise = new Test().surprise();
        System.out.println("Hello, World!", surprise);
    }

    public String surprise() {
        // TODO: Implement surprise function
    }
}


In [16]:
for i in range(5):
    test_approach_2(i)

INFO:httpx:HTTP Request: POST https://chat-1.ki-awz.iisys.de/api/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-1.ki-awz.iisys.de/api/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-1.ki-awz.iisys.de/api/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-1.ki-awz.iisys.de/api/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-1.ki-awz.iisys.de/api/chat/completions "HTTP/1.1 200 OK"


In [24]:
import re

for i in range(5):
    with open(f"test_results/approach_2_output_{i}.txt", "r") as f:
        response_content = f.read()
        
    print(f"===============\napproach_2_output_{i}.txt")
    for block in  re.finditer(r"```java\n(?P<code>.*?)\n```", response_content, re.DOTALL):
        code_block = block.group("code")
        if not isinstance(code_block, str):
            print(f"Response {i} does not contain a valid code block.")
            continue
        formatted_block = code_block.replace("\n", "\n\t")
        print(f"\t{formatted_block}")
        
        final_code = template.render(surprise_code=code_block)
        with open(f"test_results/approach_2_output_{i}.java", "w") as f:
            f.write(final_code)

approach_2_output_0.txt
	// Returns a surprise message
	return "Surprise!";
approach_2_output_1.txt
	    // This function returns a surprise message
	    return "Surprise!";
approach_2_output_2.txt
	    // TODO: Implement surprise function
	    return "Surprise!";
approach_2_output_3.txt
	// Returns a surprise message
	return "Surprise!";
approach_2_output_4.txt
	// This function returns a surprise message
	return "Surprise!";
